In [1]:
import pandas as pd

# Load and merge tables
transactions = pd.read_csv("https://cdn.enqurious.com/documents/c1783c85-e8de-4ccc-a92e-bfc37f8fbf93_extransactions.csv")
orders       = pd.read_csv("https://cdn.enqurious.com/documents/e232a410-6d01-4c69-b468-4ef821382898_exorders.csv")
products     = pd.read_csv("https://cdn.enqurious.com/documents/c2ac5069-8a28-487f-8aee-05c385222a6a_exproducts.csv")



In [2]:
import pandas as pd

# Load and merge tables (from Input 3 setup)
transactions = pd.read_csv("https://cdn.enqurious.com/documents/c1783c85-e8de-4ccc-a92e-bfc37f8fbf93_extransactions.csv")
orders       = pd.read_csv("https://cdn.enqurious.com/documents/e232a410-6d01-4c69-b468-4ef821382898_exorders.csv")
products     = pd.read_csv("https://cdn.enqurious.com/documents/c2ac5069-8a28-487f-8aee-05c385222a6a_exproducts.csv")

trans = transactions.merge(
    products[["product_id", "product_name", "category", "sub_category"]],
    left_on="Product_ID", right_on="product_id", how="left"
)
trans = trans.merge(
    orders[["order_id", "ship_mode", "order_status"]],
    left_on="Order_ID", right_on="order_id", how="left"
)

# Function definition ───────────────────────────────────────────────────────
def get_category_summary(df, category, min_discount=0.0):
    """
    Returns a one-row summary of sales metrics for a given product category.

    Parameters
    ----------
    df           : merged transactions DataFrame
    category     : str   — product category (e.g. "Electronics")
    min_discount : float — minimum discount to include (default 0.0 = all)

    Returns
    -------
    pd.DataFrame with columns: category, total_revenue, total_quantity, avg_discount
    """
    # Apply category and discount filters
    filtered = df[
        (df["category"]  == category) &
        (df["Discount"]  >= min_discount)
    ]

    # Build the summary as a single-row DataFrame
    summary = pd.DataFrame([{
        "category"       : category,
        "total_revenue"  : filtered["Sales_Amount"].sum(),
        "total_quantity" : filtered["Quantity"].sum(),
        "avg_discount"   : round(filtered["Discount"].mean(), 2)
    }])
    return summary

# Function calls ────────────────────────────────────────────────────────────
# Positional argument — uses default min_discount = 0.0
print(get_category_summary(trans, "Electronics"))

# Keyword argument — override the default to filter for ≥ 20% discount only
print(get_category_summary(trans, "Electronics", min_discount=0.2))

      category  total_revenue  total_quantity  avg_discount
0  Electronics            0.0               0           NaN
      category  total_revenue  total_quantity  avg_discount
0  Electronics            0.0               0           NaN


In [3]:
def get_category_summary(df, category, min_discount=0.0, status=None):
    """
    Returns a one-row sales summary for a given category, with optional
    discount threshold and order status filters.

    Parameters
    ----------
    df           : merged transactions DataFrame
    category     : str or None  - product category (e.g. "Furniture")
    min_discount : float        - minimum discount to include (default 0.0)
    status       : str or None  - order_status to filter on (default None = all statuses)

    Returns
    -------
    pd.DataFrame with columns: category, total_revenue, total_quantity, avg_discount
    """
    # Start with category + discount filters
    filtered = df[
        (df["category"] == category) &
        (df["Discount"] >= min_discount)
    ]

    # Apply optional status filter only when provided
    if status is not None:
        filtered = filtered[filtered["order_status"] == status]

    summary = pd.DataFrame([{
        "category"       : category,
        "total_revenue"  : filtered["Sales_Amount"].sum(),
        "total_quantity" : filtered["Quantity"].sum(),
        "avg_discount"   : round(filtered["Discount"].mean(), 2)
    }])
    return summary

# Uses default min_discount and no status filter - backward-compatible
print(get_category_summary(trans, "Furniture"))

# Keyword arguments — only delivered Furniture orders with ≥ 10% discount
print(get_category_summary(trans, "Furniture", min_discount=0.1, status="delivered"))

    category  total_revenue  total_quantity  avg_discount
0  Furniture    730281.7468            7894          0.17
    category  total_revenue  total_quantity  avg_discount
0  Furniture    461334.8743            4561          0.29


In [4]:
def filter_by_ship_mode(df, ship_mode, min_sales=0):
    """
    Filters transactions by ship mode and an optional minimum sales threshold.

    Parameters
    ----------
    df        : merged transactions DataFrame
    ship_mode : str   - shipment mode to filter on (e.g. "Premium", "Standard")
    min_sales : float - minimum sales_amount to include (default 0 = no threshold)

    Returns
    -------
    pd.DataFrame of filtered transaction rows
    """
    filtered = df[
        (df["ship_mode"]    == ship_mode) &
        (df["Sales_Amount"] >= min_sales)
    ]

    # Return a clean subset of columns with a reset index
    return filtered[[
        "TransactionID", "Order_ID", "product_name",
        "category", "Sales_Amount", "Quantity", "Discount", "ship_mode"
    ]].reset_index(drop=True)

# Positional only — returns all Premium transactions (min_sales defaults to 0)
print(filter_by_ship_mode(trans, "Premium"))

# Positional + keyword — Standard transactions worth at least $3,000
print(filter_by_ship_mode(trans, "Standard", min_sales=3000))

# Keyword arguments only — same result, order-independent
print(filter_by_ship_mode(df=trans, ship_mode="Fastrack", min_sales=1500))

     TransactionID        Order_ID  \
0               15  US-2017-168116   
1               27  CA-2015-114811   
2               79  CA-2016-129630   
3               80  CA-2014-160766   
4              104  CA-2015-139731   
..             ...             ...   
518           9792  CA-2017-121160   
519           9809  CA-2015-146829   
520           9817  US-2014-152723   
521           9818  CA-2014-112403   
522           9820  CA-2017-124114   

                                          product_name         category  \
0                                           Xerox 1967  Office Supplies   
1        Bretford CR4500 Series Slim Rectangular Table        Furniture   
2                         6" Cubicle Wall Clock, Black        Furniture   
3    SimpliFile Personal File, Black Granite, 15w x...  Office Supplies   
4        Avery Hidden Tab Dividers for Binding Systems  Office Supplies   
..                                                 ...              ...   
518      Eureka Re

In [5]:
def get_top_products(df, category, top_n=5):
    """
    Returns a ranked leaderboard of the top-N products by total sales
    within a given category.

    Parameters
    ----------
    df       : merged transactions DataFrame
    category : str — product category to filter on (e.g. "Electronics")
    top_n    : int — number of top products to return (default 5)

    Returns
    -------
    pd.DataFrame with columns: rank, product_name, category,
                               total_revenue, total_quantity
    """
    # Filter to the requested category
    filtered = df[df["category"] == category]

    # Aggregate total revenue and quantity per product, sort, take top N
    leaderboard = (
        filtered
        .groupby("product_name")
        .agg(
            total_revenue  = ("Sales_Amount", "sum"),
            total_quantity = ("Quantity",     "sum")
        )
        .reset_index()
        .sort_values("total_revenue", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # Add rank and category columns at the front
    leaderboard.insert(0, "rank",     range(1, len(leaderboard) + 1))
    leaderboard.insert(2, "category", category)

    return leaderboard

# Positional only — top 5 Electronics (uses default top_n = 5)
print(get_top_products(trans, "Electronics"))

# Positional + keyword — top 3 Furniture products
print(get_top_products(trans, "Furniture", top_n=3))

# Keyword arguments only — top 10 Office Supplies
print(get_top_products(df=trans, category="Office Supplies", top_n=10))

Empty DataFrame
Columns: [rank, product_name, category, total_revenue, total_quantity]
Index: []
   rank                                       product_name   category  \
0     1    Target 5400 Series Task Chairs for Big and Tall  Furniture   
1     2  Riverside Palais Royal Lawyers Bookcase, Royal...  Furniture   
2     3         Bretford Rectangular Conference Table Tops  Furniture   

   total_revenue  total_quantity  
0     21870.5760              39  
1     15610.9656              24  
2     12995.2915              46  
   rank                                       product_name         category  \
0     1  Fellowes PB500 Electric Punch Plastic Comb Bin...  Office Supplies   
1     2   GBC Ibimaster 500 Manual ProClick Binding System  Office Supplies   
2     3         GBC DocuBind TL300 Electric Binding System  Office Supplies   
3     4          GBC DocuBind P400 Electric Binding System  Office Supplies   
4     5   Ikea High Speed Automatic Electric Letter Opener  Office Supplies